In [0]:
# ===========================================
# Notebook Name:
# 03_build_deck_pokemon_features
#
# Purpose:
# Build Pokemon-only deck features for
# similarity and archetype analysis.
#
# Input:
# pokemon.silver.deck_cards
# pokemon.gold.deck_registry
#
# Output:
# pokemon.gold.deck_pokemon_features
#
# Grain:
# One row per deck_hash and Pokemon card name
#
# Business rules:
# - Only category = pokemon is included
# - Different printings of the same card name
#   are aggregated
# - quantity is the ML feature value
# ===========================================

In [0]:
from pyspark.sql import functions as F

DECK_CARDS_TABLE = (
    "pokemon.silver.deck_cards"
)

DECK_REGISTRY_TABLE = (
    "pokemon.gold.deck_registry"
)

DECK_POKEMON_FEATURES_TABLE = (
    "pokemon.gold.deck_pokemon_features"
)

print("Input :", DECK_CARDS_TABLE)
print("Input :", DECK_REGISTRY_TABLE)
print("Output:", DECK_POKEMON_FEATURES_TABLE)

In [0]:
deck_cards_df = spark.table(
    DECK_CARDS_TABLE
)

deck_registry_df = spark.table(
    DECK_REGISTRY_TABLE
)

print(
    "Deck card rows:",
    deck_cards_df.count(),
)

print(
    "Registered deck hashes:",
    deck_registry_df
    .select("deck_hash")
    .distinct()
    .count(),
)

In [0]:
pokemon_cards_df = (
    deck_cards_df
    .filter(
        F.col("category") == "pokemon"
    )
    .filter(
        F.col("deck_hash").isNotNull()
    )
    .filter(
        F.col("card_name").isNotNull()
    )
    .filter(
        F.trim(F.col("card_name")) != ""
    )
)

display(
    pokemon_cards_df.select(
        "deck_hash",
        "deck_code",
        "card_name",
        "quantity",
        "expansion",
        "collection_number",
    )
    .orderBy(
        "deck_hash",
        "card_name",
    )
)

In [0]:
deck_pokemon_features_df = (
    pokemon_cards_df
    .groupBy(
        "deck_hash",
        "card_name",
    )
    .agg(
        F.sum("quantity")
        .cast("int")
        .alias("quantity"),

        F.countDistinct("deck_code")
        .alias("source_deck_code_count"),

        F.sort_array(
            F.collect_set("deck_code")
        ).alias("source_deck_codes"),

        F.sort_array(
            F.collect_set("expansion")
        ).alias("expansions"),

        F.sort_array(
            F.collect_set(
                "collection_number"
            )
        ).alias("collection_numbers"),
    )
    .withColumn(
        "is_present",
        F.lit(1),
    )
    .withColumn(
        "updated_at",
        F.current_timestamp(),
    )
)

display(
    deck_pokemon_features_df
    .orderBy(
        "deck_hash",
        F.col("quantity").desc(),
        "card_name",
    )
)

In [0]:
eligible_deck_hashes_df = (
    deck_registry_df
    .select(
        "deck_hash",
        "deck_hash_version",
        "representative_deck_code",
    )
    .dropDuplicates(["deck_hash"])
)

deck_pokemon_features_df = (
    deck_pokemon_features_df
    .join(
        eligible_deck_hashes_df,
        on="deck_hash",
        how="inner",
    )
    .select(
        "deck_hash",
        "deck_hash_version",
        "representative_deck_code",
        "card_name",
        "quantity",
        "is_present",
        "source_deck_code_count",
        "source_deck_codes",
        "expansions",
        "collection_numbers",
        "updated_at",
    )
)

In [0]:
pokemon_feature_summary_df = (
    deck_pokemon_features_df
    .groupBy("deck_hash")
    .agg(
        F.sum("quantity").alias(
            "total_pokemon_cards"
        ),
        F.countDistinct("card_name").alias(
            "unique_pokemon_names"
        ),
        F.max("quantity").alias(
            "max_copies_of_one_pokemon"
        ),
    )
)

display(
    pokemon_feature_summary_df
    .orderBy(
        F.col("total_pokemon_cards").desc()
    )
)

In [0]:
duplicate_feature_df = (
    deck_pokemon_features_df
    .groupBy(
        "deck_hash",
        "card_name",
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

duplicate_count = (
    duplicate_feature_df.count()
)

if duplicate_count > 0:
    display(duplicate_feature_df)

    raise ValueError(
        f"{duplicate_count} duplicated "
        "deck/card features detected"
    )

print(
    "Validation passed: "
    "one row per deck_hash and card_name"
)

In [0]:
invalid_quantity_df = (
    deck_pokemon_features_df
    .filter(
        (F.col("quantity") <= 0)
        | (F.col("quantity") > 60)
    )
)

invalid_quantity_count = (
    invalid_quantity_df.count()
)

if invalid_quantity_count > 0:
    display(invalid_quantity_df)

    raise ValueError(
        f"{invalid_quantity_count} invalid "
        "Pokemon quantities detected"
    )

print(
    "Validation passed: "
    "Pokemon quantities are valid"
)

In [0]:
feature_deck_hashes_df = (
    deck_pokemon_features_df
    .select("deck_hash")
    .distinct()
)

missing_feature_decks_df = (
    eligible_deck_hashes_df
    .select("deck_hash")
    .join(
        feature_deck_hashes_df,
        on="deck_hash",
        how="left_anti",
    )
)

missing_feature_count = (
    missing_feature_decks_df.count()
)

if missing_feature_count > 0:
    display(missing_feature_decks_df)

    raise ValueError(
        f"{missing_feature_count} decks "
        "have no Pokemon features"
    )

print(
    "Validation passed: "
    "all registered decks have Pokemon features"
)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{DECK_POKEMON_FEATURES_TABLE}
(
    deck_hash STRING NOT NULL,
    deck_hash_version STRING NOT NULL,
    representative_deck_code STRING,
    card_name STRING NOT NULL,
    quantity INT NOT NULL,
    is_present INT NOT NULL,
    source_deck_code_count BIGINT,
    source_deck_codes ARRAY<STRING>,
    expansions ARRAY<STRING>,
    collection_numbers ARRAY<STRING>,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Pokemon-only deck features for similarity and archetype analysis'
""")

In [0]:
spark.sql(
    f"TRUNCATE TABLE "
    f"{DECK_POKEMON_FEATURES_TABLE}"
)

(
    deck_pokemon_features_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        DECK_POKEMON_FEATURES_TABLE
    )
)

print(
    "Gold deck_pokemon_features rebuilt."
)

In [0]:
display(
    spark.table(
        DECK_POKEMON_FEATURES_TABLE
    )
    .orderBy(
        "deck_hash",
        F.col("quantity").desc(),
        "card_name",
    )
)

In [0]:
display(
    spark.table(
        DECK_POKEMON_FEATURES_TABLE
    )
    .groupBy("deck_hash")
    .agg(
        F.sum("quantity").alias(
            "total_pokemon_cards"
        ),
        F.countDistinct("card_name").alias(
            "unique_pokemon_names"
        ),
    )
    .orderBy(
        F.col("total_pokemon_cards").desc()
    )
)

In [0]:
pokemon_feature_matrix_df = (
    spark.table(
        DECK_POKEMON_FEATURES_TABLE
    )
    .groupBy("deck_hash")
    .pivot("card_name")
    .sum("quantity")
    .fillna(0)
)

display(pokemon_feature_matrix_df)

In [0]:
feature_count = (
    spark.table(
        DECK_POKEMON_FEATURES_TABLE
    )
    .select("card_name")
    .distinct()
    .count()
)

deck_count = (
    spark.table(
        DECK_POKEMON_FEATURES_TABLE
    )
    .select("deck_hash")
    .distinct()
    .count()
)

print("Deck count   :", deck_count)
print("Feature count:", feature_count)